In [ ]:
# Practice Problems Day 3

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 3 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron, chapter 4, the section on **Regularized Linear Models**

GOALS:

1. See the bias variance tradeoff happen in numbers, not just in words.
2. Get comfortable with Ridge, Lasso, and Elastic Net, and with picking an alpha.
3. Notice that scaling and leakage are not side details, they change your answers.

**One warning you can ignore.** With degree 12 polynomial features, Lasso will often
print `ConvergenceWarning: Objective did not converge` even when you pass
`max_iter=100000`. That is because high powers of $x$ are almost perfectly related to
each other, which makes the problem hard for the solver to finish cleanly. The answers
are still usable. Any *other* warning you should read and deal with.

Several of these problems ask you to **guess before you run the code**. Write the
guess down, then run it. Do not go back and change your guess to match the answer.
A wrong guess you can explain teaches you more than a right one!

---

## Problem 1: Set up your own data

Build a dataset the way we did in class, but use a **cubic** true function instead of
a parabola. Something like

$$y = 0.3x^3 - x + 2 + \text{noise}$$

Use 60 points with $x$ between -3 and 3, and add normal noise with a standard
deviation of 1.0.

**1a.** Make the data and plot it.

**1b.** Split it into a training set and a validation set, 70 percent training.

**1c.** How many points ended up in each set?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(seed=7)

# 1a. a cubic instead of a parabola
m = 60
X_all = 6 * rng.random((m, 1)) - 3
y_all = 0.3 * X_all**3 - X_all + 2 + rng.standard_normal((m, 1))

plt.figure(figsize=(6, 4))
plt.plot(X_all, y_all, "b.", markersize=8)
plt.xlabel("$x$"); plt.ylabel("$y$", rotation=0)
plt.grid(); plt.title("A cubic plus noise, 60 points")
plt.show()

# 1b. split it
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_all, test_size=0.3, random_state=7)

# 1c.
print("training points  :", len(X_train))
print("validation points:", len(X_valid))

**1c.** 42 training points and 18 validation points.

NOTE - eighteen validation points is very few. Any RMSE you compute in this assignment has a lot of wobble in it. Keep that in mind, it is exactly what Day 4 is about.

---

## Problem 2: Ridge and the alpha sweep

Build degree 12 polynomial features from your training data, scale them, and fit
Ridge for a range of alpha values.

**2a.** Before you run anything, guess: will the best alpha for this data be bigger
or smaller than the 1.19 we got in class? Why? (Hint: you have 42 training points
instead of 140, and a cubic instead of a parabola.)

**2b.** Sweep alpha over `np.logspace(-4, 4, 30)`. Plot the training RMSE and the
validation RMSE against alpha on a log x axis.

**2c.** What is your best alpha, and what validation RMSE does it get? Was your guess
in 2a right?

**2d.** The noise you added has a standard deviation of 1.0. How close is your best
model to that floor? Is there much room left to improve?

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error

# Step 1: features from the TRAINING data only
poly_features = PolynomialFeatures(degree=12, include_bias=False)
X_train_poly = poly_features.fit_transform(X_train)
X_valid_poly = poly_features.transform(X_valid)

# Step 2: scale, again learned from training only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_valid_scaled = scaler.transform(X_valid_poly)

# Step 3: sweep alpha
alpha_list = np.logspace(-4, 4, 30)
train_rmse = []
valid_rmse = []

for alpha in alpha_list:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)
    train_rmse.append(root_mean_squared_error(y_train, model.predict(X_train_scaled)))
    valid_rmse.append(root_mean_squared_error(y_valid, model.predict(X_valid_scaled)))

best = np.argmin(valid_rmse)

print("best alpha      =", round(alpha_list[best], 4))
print("  training RMSE   =", round(train_rmse[best], 3))
print("  validation RMSE =", round(valid_rmse[best], 3))
print()
print("smallest alpha tried,", round(alpha_list[0], 5))
print("  training RMSE   =", round(train_rmse[0], 3))
print("  validation RMSE =", round(valid_rmse[0], 3))
print()
print("largest alpha tried,", round(alpha_list[-1], 1))
print("  training RMSE   =", round(train_rmse[-1], 3))
print("  validation RMSE =", round(valid_rmse[-1], 3))

plt.figure(figsize=(6.5, 4.5))
plt.semilogx(alpha_list, train_rmse, "r-+", linewidth=2, label="training set")
plt.semilogx(alpha_list, valid_rmse, "b-", linewidth=2.5, label="validation set")
plt.axvline(alpha_list[best], color="k", linestyle=":",
            label=f"best alpha = {alpha_list[best]:.3f}")
plt.xlabel("alpha (log scale)"); plt.ylabel("RMSE")
plt.legend(); plt.grid()
plt.title("Degree 12 Ridge on 42 training points")
plt.show()

**2a.** My guess: **bigger**. With only 42 training points a degree 12 model has a lot more freedom compared to the amount of data than it did with 140 points, so it needs a heavier penalty to keep it in line. Either guess is fine if your reason is about the amount of data or how flexible the model is.

**2b.** The plot above. The validation curve has the U shape: too little regularization on the left, too much on the right.

**2c.** My best alpha is about **2.59**, with a validation RMSE of **0.989**. That is bigger than the 1.19 from class, so the guess was right.

**2d.** The noise has a standard deviation of 1.0 and my best validation RMSE is 0.989. That is right at the noise level, so there is basically no room left. The honest answer is "this is about as good as it gets, stop tuning."

---

## Problem 3: Tune alpha the wrong way

In Problem 2 you picked alpha by looking at the **validation** RMSE. This problem asks
you to do it wrong instead, so you can see what wrong looks like.

**3a.** Using the arrays you already built in Problem 2, find the alpha that gives the
lowest **training** RMSE. What alpha is it, and what training RMSE does it get?

**3b.** That training RMSE is a better number than the one your Problem 2 model got.
Now compute the **validation** RMSE of this training-chosen model and compare it to
your Problem 2 answer. Which model would you rather hand to a customer?

**3c.** Look at the shape of your training RMSE curve from Problem 2. Explain why
picking alpha by training error will **always** choose the smallest alpha you tried, no
matter what data you have. This is a property of the cost function, not of your dataset.

**3d.** Here is the hard part. Suppose you had reported the training RMSE from 3a to
your boss as "how well this model will do on new customers." In what direction would
you be lying, and who ends up surprised?

**3e.** Now a subtler version. Rebuild your features and your scaler using **all** of
your data before you split it:

```python
# THE WRONG WAY, do this only for this problem
X_poly_all = poly_features.fit_transform(X_all)
X_scaled_all = scaler.fit_transform(X_poly_all)
# and only then split into train and validation
```

Report the validation RMSE and compare it to Problem 2. You will probably find the
difference is tiny, possibly zero. Given that, explain why this is still something you
should never do. What is actually wrong with the number, if it is not that the number is
far off?

In [ ]:
# 3a. pick alpha by TRAINING error instead of validation error
worst_choice = np.argmin(train_rmse)

print("alpha chosen by training error =", round(alpha_list[worst_choice], 5))
print("  its training RMSE   =", round(train_rmse[worst_choice], 3))
print("  its validation RMSE =", round(valid_rmse[worst_choice], 3))
print()
print("alpha chosen by validation error =", round(alpha_list[best], 4))
print("  its training RMSE   =", round(train_rmse[best], 3))
print("  its validation RMSE =", round(valid_rmse[best], 3))

# 3c. is the training curve really monotone?
print("\ntraining RMSE only ever increases with alpha:",
      bool(np.all(np.diff(train_rmse) >= -1e-12)))

# 3e. the subtler leak: fit features and scaler on everything, then split
poly_leaky = PolynomialFeatures(degree=12, include_bias=False)
X_poly_all = poly_leaky.fit_transform(X_all)

scaler_leaky = StandardScaler()
X_scaled_all = scaler_leaky.fit_transform(X_poly_all)

X_tr_leak, X_va_leak, y_tr_leak, y_va_leak = train_test_split(
    X_scaled_all, y_all, test_size=0.3, random_state=7)

best_alpha = alpha_list[best]

leaky_model = Ridge(alpha=best_alpha)
leaky_model.fit(X_tr_leak, y_tr_leak)
rmse_leaky = root_mean_squared_error(y_va_leak, leaky_model.predict(X_va_leak))

honest_model = Ridge(alpha=best_alpha)
honest_model.fit(X_train_scaled, y_train)
rmse_honest = root_mean_squared_error(y_valid, honest_model.predict(X_valid_scaled))

print("\nsame alpha, scaler fit two different ways:")
print("  leaky  validation RMSE =", round(rmse_leaky, 4))
print("  honest validation RMSE =", round(rmse_honest, 4))
print("  difference             =", round(rmse_honest - rmse_leaky, 4))

**3a.** Picking by training error chooses alpha = 0.0001, the smallest value in the sweep, with a training RMSE of 0.661.

**3b.** 0.661 looks better than the 0.734 training RMSE of the Problem 2 model. But its validation RMSE is 1.108, against 0.989 for the Problem 2 model, so it is worse on data it has not seen. You hand the customer the Problem 2 model.

**3c.** This one is about the cost function, not your data. Alpha = 0 is the value that makes the training error as small as possible, because then the model only cares about fitting the training points. Every bigger alpha spends some of the model's effort keeping the weights small instead. So the training RMSE can only go up as alpha goes up (the code checks this, it prints `True`), and "lowest training error" always means "smallest alpha you tried."

**3d.** The training RMSE is too **optimistic**, so you are promising better performance than the model will deliver. The one who ends up surprised is the boss (and the customers), when the model does noticeably worse on new data than the number in your report.

**3e.** On this data the leak makes almost **no difference**, 0.9887 against 0.989. That is a real result, and I would rather you see it than be told a scary story.

So what is wrong? Leakage does not always show up as a bad number. It shows up as a number you **cannot trust**, and you cannot tell by looking at it how much it is off. Here the leak was small because a mean and standard deviation computed from 60 points are close to ones computed from 42. Change the setup, say one wild outlier that lands in the validation set, and the same mistake moves the number a lot. You do not get to know ahead of time which situation you are in, so you never do it.

BEWARE - if you got a **big** difference here, check your Problem 2 code. It probably means your Problem 2 was already leaking.

---

## Problem 4: Lasso finds the answer

Fit Lasso to your degree 12 features. Remember to pass `max_iter=100000` so it
converges.

**4a.** Try at least four values of alpha. For each one, print the alpha, how many
weights are not zero, and the validation RMSE.

**4b.** Find an alpha where Lasso keeps only a few weights. **Which powers of $x$
does it keep?** Print `lasso_model.coef_` and read it. Did it find your cubic?

**4c.** Make a bar chart of the weights for your best Lasso model and your best Ridge
model, one above the other, the way we did in the notes.

**4d.** In one or two sentences, what is different about the two charts, and what
does that tell you about what the two methods are doing?

In [ ]:
from sklearn.linear_model import Lasso

print("Lasso on degree 12 features:")
lasso_results = {}
for alpha in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    model = Lasso(alpha=alpha, max_iter=100000)
    model.fit(X_train_scaled, y_train.ravel())

    n_used = np.sum(np.abs(model.coef_) > 1e-8)
    rmse = root_mean_squared_error(y_valid, model.predict(X_valid_scaled))
    lasso_results[alpha] = (n_used, rmse, model)
    print(f"  alpha = {alpha:<8}  weights used = {n_used:2d}/12   validation RMSE = {rmse:.3f}")

# 4b. which powers survive at a sparse setting?
best_lasso_alpha = min(lasso_results, key=lambda a: lasso_results[a][1])
n_used, rmse, best_lasso = lasso_results[best_lasso_alpha]
print(f"\nbest Lasso alpha = {best_lasso_alpha}, keeps {n_used} weights, RMSE {rmse:.3f}")
print("\nthe weights it kept:")
for power, w in enumerate(best_lasso.coef_, start=1):
    if abs(w) > 1e-8:
        print(f"  x^{power:<3} weight = {w: .3f}")

In [ ]:
# 4c. bar charts, Ridge above Lasso
best_ridge = Ridge(alpha=best_alpha)
best_ridge.fit(X_train_scaled, y_train)

powers = np.arange(1, 13)
fig, axes = plt.subplots(2, 1, figsize=(7.5, 5.5), sharex=True)

for ax, coefs, name in zip(axes,
        [best_ridge.coef_.ravel(), best_lasso.coef_.ravel()],
        [f"Ridge, alpha = {best_alpha:.3f}", f"Lasso, alpha = {best_lasso_alpha}"]):
    ax.bar(powers, coefs, color="steelblue")
    ax.axhline(0, color="k", linewidth=0.8)
    ax.set_title(name); ax.grid(axis="y"); ax.set_ylabel("weight")

axes[-1].set_xlabel("which power of $x$")
axes[-1].set_xticks(powers)
plt.tight_layout()
plt.show()

print("total size of the weights:")
print("  Ridge:", round(np.abs(best_ridge.coef_).sum(), 2))
print("  Lasso:", round(np.abs(best_lasso.coef_).sum(), 2))

**4a.** The printout above: as alpha goes up, Lasso uses fewer and fewer weights.

**4b.** My best Lasso alpha was 0.1. It keeps 3 of the 12 weights, with a validation RMSE of 0.985. The three it kept were $x^5$, $x^7$ and $x^{11}$.

It did **not** keep $x$ and $x^3$, the two terms that are actually in the true function! That is not a mistake in the code. On the interval we are using, high powers of $x$ are almost perfectly related to each other, so $x^5$ can stand in for $x^3$. When Lasso has a group of features that carry almost the same information, it keeps one and zeros out the rest, and which one it keeps is close to random. The model still predicts well, 0.985 against Ridge's 0.989.

Keep this in mind for Problem 6. If your Lasso did keep $x$ and $x^3$, that is fine too. A different split gives a different pick.

**4c.** The bar charts above.

**4d.** Ridge: every bar is there and they are all short, total size 2.44. Lasso: most bars are exactly zero, total size 1.65. Ridge shrinks every weight, Lasso deletes some of them. A simpler model is easier to explain, but 4b shows the cost: the weights Lasso keeps are not always the "real" ones.

---

## Problem 5: A real dataset

Enough synthetic data. Load the California housing dataset, which is built into
scikit-learn.

```python
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
X = housing.data      # 8 features
y = housing.target    # median house value, in hundreds of thousands of dollars
```

The first time you call `fetch_california_housing` it downloads about 1.4 MB and
caches it, so you need to be online for that one call. After that it is local.

**5a.** How many rows and how many features? What are the feature names? Are they on
similar scales? Show the evidence, do not just say yes or no.

**5b.** Split into training and validation. Fit **three** models on the training
data: plain `LinearRegression`, `Ridge`, and `Lasso`. Scale the features first.
Report the validation RMSE of each.

**5c.** Which model won? By how much?

**5d.** Look at the Lasso weights. Did it zero anything out? If so, which features,
and does that surprise you given what the features are?

**5e.** Your validation RMSE is in units of hundreds of thousands of dollars. Convert
your best number into actual dollars and write one sentence a real estate agent would
understand.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression

housing = fetch_california_housing(as_frame=True)
X_h = housing.data
y_h = housing.target

# 5a.
print("shape:", X_h.shape)
print("features:", list(X_h.columns))
print()
print(X_h.describe().T[["mean", "std", "min", "max"]].round(2))

In [ ]:
# 5b.
Xh_train, Xh_valid, yh_train, yh_valid = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42)

scaler_h = StandardScaler()
Xh_train_scaled = scaler_h.fit_transform(Xh_train)
Xh_valid_scaled = scaler_h.transform(Xh_valid)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge, alpha=1":   Ridge(alpha=1.0),
    "Lasso, alpha=0.01": Lasso(alpha=0.01, max_iter=100000),
}

scores = {}
for name, model in models.items():
    model.fit(Xh_train_scaled, yh_train)
    rmse = root_mean_squared_error(yh_valid, model.predict(Xh_valid_scaled))
    scores[name] = rmse
    print(f"  {name:<20} validation RMSE = {rmse:.4f}")

# 5d.
lasso_h = models["Lasso, alpha=0.01"]
print("\nLasso weights on the housing data:")
for name, w in zip(X_h.columns, lasso_h.coef_):
    flag = "   <-- ZEROED OUT" if abs(w) < 1e-8 else ""
    print(f"  {name:<12} {w: .4f}{flag}")

In [ ]:
# 5e. convert to dollars
best_name = min(scores, key=scores.get)
best_rmse = scores[best_name]
print("best model:", best_name)
print("validation RMSE:", round(best_rmse, 4), "hundreds of thousands of dollars")
print("in dollars:", "$" + format(int(best_rmse * 100000), ","))

**5a.** 20640 rows and 8 features. They are on very different scales, and the `describe()` table is the evidence: `Population` goes up to 35682 while `AveBedrms` sits near 1.

**5b.** The printout above. I scaled the features first, fit on the training data only.

**5c.** Lasso won, 0.7404 against 0.7456 for both `LinearRegression` and `Ridge`. That is a tiny difference, about 0.005, so really the three models are tied.

**5d.** Lasso zeroed out `Population`. That makes sense: the number of people in a block group says little about house prices once you know the income and the location.

**5e.** 0.7404 hundreds of thousands of dollars is about **$74,000**. For a real estate agent: "The model's price guess for a neighborhood is typically off by about $74,000."

BEWARE - do not forget to multiply by 100,000. An error of "0.74 dollars" would be a pretty amazing model!

---

## Problem 6: A judgment call

No code for this one. You are handed a model and a report.

A teammate built a model to predict which patients will need to be readmitted to the
hospital within 30 days. They used Lasso with a big alpha, they are proud that it
only kept 4 features out of 60, and they say this makes the model easy to explain to
doctors. Their validation RMSE is 0.31. The model with no regularization got 0.28.

**6a.** Is the sparse model better or worse? On what grounds?

**6b.** Give one good reason to prefer their sparse model anyway, even though its
error is higher.

**6c.** Give one reason to be nervous about it. Think about what Lasso does when two
features carry similar information, and what that means if one of the 56 discarded
features was clinically important.

**6d.** What would you want to know before you agreed to put either model into a
hospital?

*There is more than one reasonable answer. I care about your reasoning.*

**6a.** Worse on error, 0.31 against 0.28. That is the only thing the numbers tell you, and it is in the wrong direction for the sparse model.

**6b.** A doctor can look at 4 features and check that the model makes sense, which they cannot do with 60. A model people trust and actually use beats a slightly better one they ignore. Fewer features also means fewer things to collect, fewer places for missing data, and less to maintain.

**6c.** Remember Problem 4b! Lasso picks almost randomly among features that carry similar information, so a feature being dropped is **not** evidence that it does not matter. In a hospital, something clinically important could disappear from the model for a statistical reason, the score stays fine, and nobody notices. Also, 0.31 against 0.28 on one validation set might not be a real difference at all.

**6d.** Some good things to ask: how big was the validation set? Does the comparison hold up with cross validation? Which 56 features were dropped, and does a doctor agree they are unimportant? How are the errors spread across different groups of patients, not just on average? And what happens to a patient the model gets wrong? That last one is the Day 1 question coming back.